# Eksperimen #1 — Early stopping pada val macro-F1

**Perubahan tunggal** dari baseline (`05_train_*`): sinyal yang dipantau early-stopping
diganti dari `val_loss` → **val macro-F1**. Semua yang lain (fitur, arsitektur head,
seed, split) identik, supaya efeknya bisa diisolasi.

## Kenapa

Di D2, `val_loss` minimum ada di **epoch 1** karena model cepat *overconfident* (loss
menghukum keyakinan-salah, padahal argmax terus membaik). Akibatnya early-stopping
memulihkan bobot epoch-1 yang belum matang. Metrik yang kita pedulikan adalah macro-F1,
jadi seharusnya *itu* yang dipantau.

**Baseline pembanding:** D1 macro-F1 `0.9643` · D2 macro-F1 `0.7639`.

> Ini eksperimen terpisah: menambah baris `*_esf1` ke `experiments.csv`, baseline tidak
> ditimpa.

In [1]:
import json
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import f1_score, accuracy_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ML_DIR = ROOT / "ml"


def load_dataset(dataset: str):
    """Rakit X/y train-val-test dari cache + split, standardisasi (statistik train)."""
    data = np.load(ML_DIR / "cache" / f"yamnet_{dataset}.npz", allow_pickle=True)
    emb_by_file = {fn: e for fn, e in zip(data["filename"], data["emb"])}
    classes = sorted(set(data["label"]))
    idx = {c: i for i, c in enumerate(classes)}

    def split(name):
        df = pd.read_csv(ML_DIR / f"split_{dataset}_{name}.csv")
        X = np.stack([emb_by_file[fn] for fn in df.filename]).astype(np.float32)
        y = df.label.map(idx).to_numpy()
        return X, y

    Xtr, ytr = split("train"); Xva, yva = split("val"); Xte, yte = split("test")

    mean = Xtr.mean(0, keepdims=True); std = Xtr.std(0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)               # kolom near-konstan tak diskalakan
    Xtr = (Xtr - mean) / std; Xva = (Xva - mean) / std; Xte = (Xte - mean) / std
    return classes, (Xtr, ytr), (Xva, yva), (Xte, yte)


print("helper siap. Kelas per dataset:")
for ds in ("d1", "d2"):
    classes, *_ = load_dataset(ds)
    print(f"  {ds}: {classes}")

helper siap. Kelas per dataset:
  d1: ['ambulance', 'firetruck', 'traffic']
  d2: ['ambulance', 'firetruck', 'police', 'traffic']


## Callback: hitung val macro-F1 tiap epoch

Keras tidak memantau macro-F1 secara langsung, jadi kita hitung sendiri via sklearn di
akhir tiap epoch dan taruh ke `logs['val_macro_f1']`. `EarlyStopping` lalu memantaunya
dengan `mode='max'`. Urutan callback penting — kalkulator F1 harus **sebelum**
`EarlyStopping` agar log-nya sudah terisi.

In [2]:
class ValMacroF1(tf.keras.callbacks.Callback):
    """Hitung macro-F1 di val tiap epoch, simpan ke logs untuk dipantau callback lain."""
    def __init__(self, X_val, y_val):
        super().__init__()
        self.X_val, self.y_val = X_val, y_val

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        y_pred = self.model.predict(self.X_val, verbose=0).argmax(axis=1)
        logs["val_macro_f1"] = f1_score(self.y_val, y_pred, average="macro")


def build_head(input_dim, n_classes):
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(n_classes, activation="softmax"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

## Jalankan pada D1 & D2

Satu fungsi, dipanggil untuk kedua dataset. Menyimpan artefak `*_esf1` dan menambah baris
ke `experiments.csv`.

In [3]:
def run_experiment(dataset: str):
    tf.keras.utils.set_random_seed(SEED)               # ulang seed tiap run
    classes, (Xtr, ytr), (Xva, yva), (Xte, yte) = load_dataset(dataset)

    model = build_head(Xtr.shape[1], len(classes))
    callbacks = [
        ValMacroF1(Xva, yva),                          # HARUS pertama
        tf.keras.callbacks.EarlyStopping(
            monitor="val_macro_f1", mode="max",
            patience=20, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_macro_f1", mode="max",
            factor=0.5, patience=8, min_lr=1e-5),
    ]
    hist = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                     epochs=200, batch_size=32, callbacks=callbacks, verbose=0)

    best_epoch = int(np.argmax(hist.history["val_macro_f1"])) + 1
    best_val_f1 = float(np.max(hist.history["val_macro_f1"]))

    y_pred = model.predict(Xte, verbose=0).argmax(axis=1)
    macro_f1 = f1_score(yte, y_pred, average="macro")
    acc = accuracy_score(yte, y_pred)

    # simpan artefak
    exp_id = f"{dataset}_yamnet_mlp_esf1"
    art = ML_DIR / "artifacts" / exp_id
    art.mkdir(parents=True, exist_ok=True)
    model.save(art / "model.keras")
    (art / "metrics.json").write_text(json.dumps({
        "test_macro_f1": float(macro_f1), "test_accuracy": float(acc),
        "per_class_f1": dict(zip(classes, f1_score(yte, y_pred, average=None).tolist())),
        "best_epoch": best_epoch, "best_val_macro_f1": best_val_f1,
        "epochs_ran": len(hist.history["loss"]),
    }, indent=2))

    pd.DataFrame([{
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "exp_id": exp_id, "dataset": dataset, "backbone": "yamnet",
        "head": "mlp_256_128_esf1", "n_train": len(ytr), "n_test": len(yte),
        "best_epoch": best_epoch, "test_macro_f1": round(macro_f1, 4),
        "test_accuracy": round(acc, 4),
    }]).to_csv(ML_DIR / "experiments.csv", mode="a", header=False, index=False)

    print(f"[{dataset}] epochs_ran={len(hist.history['loss'])} · "
          f"best_epoch={best_epoch} (val_f1={best_val_f1:.4f})")
    print(f"[{dataset}] TEST macro-F1={macro_f1:.4f} · acc={acc:.4f}")
    print(classification_report(yte, y_pred, target_names=classes, digits=3))
    return {"dataset": dataset, "macro_f1": macro_f1, "best_epoch": best_epoch}


results = [run_experiment(ds) for ds in ("d1", "d2")]

[d1] epochs_ran=26 · best_epoch=6 (val_f1=0.9883)
[d1] TEST macro-F1=0.9643 · acc=0.9647
              precision    recall  f1-score   support

   ambulance      0.963     0.929     0.945        28
   firetruck      0.931     0.964     0.947        28
     traffic      1.000     1.000     1.000        29

    accuracy                          0.965        85
   macro avg      0.965     0.964     0.964        85
weighted avg      0.965     0.965     0.965        85



[d2] epochs_ran=28 · best_epoch=8 (val_f1=0.7669)
[d2] TEST macro-F1=0.7725 · acc=0.7750
              precision    recall  f1-score   support

   ambulance      0.771     0.638     0.698        58
   firetruck      0.623     0.768     0.688        56
      police      0.780     0.697     0.736        66
     traffic      0.938     1.000     0.968        60

    accuracy                          0.775       240
   macro avg      0.778     0.776     0.772       240
weighted avg      0.780     0.775     0.774       240



## Perbandingan dengan baseline

In [4]:
baseline = {"d1": 0.9643, "d2": 0.7639}
print(f"{'dataset':8s} {'baseline (val_loss)':>20s} {'esf1 (val macro-F1)':>20s} {'delta':>8s}")
for r in results:
    ds = r["dataset"]; new = r["macro_f1"]; old = baseline[ds]
    print(f"{ds:8s} {old:>20.4f} {new:>20.4f} {new-old:>+8.4f}")

print("\n--- experiments.csv ---")
print(pd.read_csv(ML_DIR / "experiments.csv").to_string(index=False))

dataset   baseline (val_loss)  esf1 (val macro-F1)    delta
d1                     0.9643               0.9643  -0.0000
d2                     0.7639               0.7725  +0.0086

--- experiments.csv ---
          timestamp             exp_id dataset backbone             head  n_train  n_test  best_epoch  test_macro_f1  test_accuracy
2026-07-24T19:27:16      d1_yamnet_mlp      d1   yamnet      mlp_256_128      426      85          15         0.9643         0.9647
2026-07-24T19:27:50      d2_yamnet_mlp      d2   yamnet      mlp_256_128     1195     240           1         0.7639         0.7667
2026-07-24T19:54:56 d1_yamnet_mlp_esf1      d1   yamnet mlp_256_128_esf1      426      85           6         0.9643         0.9647
2026-07-24T19:55:04 d2_yamnet_mlp_esf1      d2   yamnet mlp_256_128_esf1     1195     240           8         0.7725         0.7750


---

**Interpretasi:** kalau D2 naik signifikan, sinyal early-stopping memang penyebab utama —
model epoch-1 tadi memang terlalu dini. Kalau D2 nyaris tak berubah, berarti langit-langit
0.76 itu masalah **diskriminasi** (sirine mirip + noisy), bukan pemilihan epoch — dan
senjata berikutnya (augmentasi / fine-tune) yang diperlukan.